# Script 1 and 2 - Search Strategies Comparison

#### Objective
This notebook indexes CV data into a qdrant vector database using 3 different search strategies stored in separate in-memory collections:
1. **BM25**
2. **Vector Embedding**
3. **SPLADE**: It's an sparse model
3. **Hybrid with Reciprocal Rank fusion**
4. **Hybrid with Relative Score Fusion**

Search quality across all 4 strategies is evaluated against the ground truth matrix using Hit Rate, MRR@2, MRR@3, Precision, Recall, and NDCG@10.

### 1. Initializing a local client

Initializing Qdrant in  mode for fast testing.

In [1]:
from qdrant_client import QdrantClient, models

client = QdrantClient(":memory:")

### 2. Loading CV Data and Models

In [2]:
import os
import json
from dotenv import load_dotenv

load_dotenv("../.env")
HUGGING_FACE_API_KEY = os.getenv("HUGGING_FACE_API_KEY")

# Load CV data
with open("../data/cv_extracted_info_eng.json", "r") as file:
    cv_data = json.load(file)

DENSE_MODEL = "microsoft/harrier-oss-v1-0.6b" # Embedding model
BM25_MODEL = "Qdrant/bm25"
SPARSE_MODEL = "prithivida/Splade_PP_en_v1"

Getting dense vector

In [3]:
from sentence_transformers import SentenceTransformer

dense_model = SentenceTransformer(DENSE_MODEL, token=HUGGING_FACE_API_KEY)

data_points = []

def serialize_cv_for_embedding(cv: dict) -> str:
    """Formats CV text specifically for optimal embedding model vectorization."""
    certifications = ", ".join(cv.get("certifications", [])) or "None"
    skills = ", ".join(cv.get("skills", [])) or "None"
    languages = ", ".join(cv.get("languages", [])) or "None"
    
    experience = "\n".join([f"- {exp}" for exp in cv.get("experience", [])])
    education = "\n".join([f"- {edu}" for edu in cv.get("education", [])])

    text = f"""Candidate Name: {cv.get("name")}
Profession: {cv.get("profession")}
Seniority Level: {cv.get("seniority_level")}
Location: {cv.get("location")}
Years of Experience: {cv.get("experience_years")}
Languages: {languages}

About Me:
{cv.get("about_me", "")}

Skills:
{skills}

Certifications:
{certifications}

Work Experience:
{experience}

Education:
{education}
""".strip()
    
    return text

for cv in cv_data:
    cv_text = serialize_cv_for_embedding(cv)
    data_points.append({
        "chunk": cv_text,
        "cv": cv
    })

embeddings = dense_model.encode([point["chunk"] for point in data_points])
print(f"embeddings.shape: {embeddings.shape}")
embeddings = embeddings.tolist()

/Users/col-ae-068/Documents/personal-projects/ai-eng-course/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 6328.34it/s]


embeddings.shape: (108, 1024)
